In [ ]:
# --- FIX NUMPY COMPATIBILITY ---
print("🔧 Downgrading NumPy to fix binary incompatibility...")
!pip install "numpy<2.0" "opencv-python-headless<4.10"
print("\n✅ Libraries patched.")

🔧 Downgrading NumPy to fix binary incompatibility...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 50.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.90
    Uninstalling opencv-python-headless-4.13.0.90:
      Successfully uninstalled opencv-python-headless-4.13.0.90



✅ Libraries patched.
⚠️ CRITICAL STEP: You must now RESTART THE RUNTIME.
👉 Go to the top menu: Runtime -> Restart Session (or Restart Runtime).
👉 Then run Step 2 below.


In [ ]:
# --- BATCH PROCESS: ONE VIDEO PER SUBFOLDER ---
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from glob import glob

input_root = "/content/drive/MyDrive/EgoDex_Data/input_video/video_learning_samples"
output_root = "/content/drive/MyDrive/EgoDex_Data/processed_data"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if 'depth_model' not in locals():
    print("⬇️ Loading Depth Anything V2 (Large)...")
    %cd /content/Depth-Anything-V2
    from depth_anything_v2.dpt import DepthAnythingV2

    # Correct Config for V2 Large
    model_configs = {
        'vitl': {
            'encoder': 'vitl',
            'features': 256,
            'out_channels': [256, 512, 1024, 1024]
        }
    }
    depth_model = DepthAnythingV2(**model_configs['vitl'])
    depth_model.load_state_dict(torch.load('checkpoints/depth_anything_v2_vitl.pth', map_location='cpu'))
    depth_model = depth_model.to(device).eval()
    print("✅ Model Loaded.")

subfolders = [f.path for f in os.scandir(input_root) if f.is_dir()]

print(f"📂 Found {len(subfolders)} categories: {[os.path.basename(s) for s in subfolders]}")

for folder in subfolders:
    folder_name = os.path.basename(folder)
    print(f"\n--- Processing Category: {folder_name} ---")

    # Find the first video in this folder (mp4 or mov)
    video_files = glob(os.path.join(folder, "*.mp4")) + glob(os.path.join(folder, "*.mov"))

    if not video_files:
        print(f"⚠️ No videos found in {folder_name}, skipping.")
        continue

    # Pick the first video found (e.g., '1.mp4')
    target_video = video_files[0]
    print(f"   🎬 Selected Video: {os.path.basename(target_video)}")

    category_out_dir = os.path.join(output_root, folder_name)
    rgb_dir = os.path.join(category_out_dir, "rgb")
    depth_dir = os.path.join(category_out_dir, "depth")

    os.makedirs(rgb_dir, exist_ok=True)
    os.makedirs(depth_dir, exist_ok=True)

    # Process the Video
    cap = cv2.VideoCapture(target_video)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        print("   ❌ Error: Video load failed.")
        continue

    print(f"   ▶️ Extracting {total_frames} frames...")

    for i in tqdm(range(total_frames), desc=f"   {folder_name}"):
        ret, frame = cap.read()
        if not ret: break

        frame_name = f"{i:06d}.png" # 000000.png
        cv2.imwrite(os.path.join(rgb_dir, frame_name), frame)

        depth = depth_model.infer_image(frame)

        depth_16bit = (depth * 1000).astype(np.uint16)
        cv2.imwrite(os.path.join(depth_dir, frame_name), depth_16bit)

    cap.release()
    print(f"   ✅ Finished {folder_name}")

print("\n🎉 ALL CATEGORIES PROCESSED!")

📂 Found 5 categories: ['open_close', 'add_remove_lid', 'basic_pick_and_place', 'assemble_disassemble_furniture_bench_stool', 'insert_remove']

--- Processing Category: open_close ---
   🎬 Selected Video: 1.mp4
   ▶️ Extracting 171 frames...


   open_close: 100%|██████████| 171/171 [01:08<00:00,  2.50it/s]


   ✅ Finished open_close

--- Processing Category: add_remove_lid ---
   🎬 Selected Video: 0.mp4
   ▶️ Extracting 94 frames...


   add_remove_lid: 100%|██████████| 94/94 [00:38<00:00,  2.43it/s]


   ✅ Finished add_remove_lid

--- Processing Category: basic_pick_and_place ---
   🎬 Selected Video: 1.mp4
   ▶️ Extracting 160 frames...


...

   ✅ Finished basic_pick_and_place

--- Processing Category: assemble_disassemble_furniture_bench_stool ---
   🎬 Selected Video: 14.mp4
   ▶️ Extracting 120 frames...


   assemble_disassemble_furniture_bench_stool: 100%|██████████| 120/120 [00:49<00:00,  2.43it/s]


   ✅ Finished assemble_disassemble_furniture_bench_stool

--- Processing Category: insert_remove ---
   🎬 Selected Video: 3.mp4
   ▶️ Extracting 203 frames...


   insert_remove: 100%|██████████| 203/203 [01:23<00:00,  2.43it/s]

   ✅ Finished insert_remove

🎉 ALL CATEGORIES PROCESSED!
